# PIPELINE CREATION, RETRAINING AND EXECUTION SCRIPTS

## PRODUCTION PIPELINE CREATION & INITIAL EXPORT

In [9]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

# --- 1. ENCAPSULATED FEATURE ENGINEER (The "Russian Doll") ---
class SmartPortFeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self):
        # The 11 features required by the XGBoost model
        self.selected_features = [
            'day_of_week', 'rolling_mean_sog', 'hdg', 'movement_stability', 
            'cog', 'speed_delta', 'heading_change', 'imo_te', 
            'reporting_interval_min', 'time_since_last_position_min', 'arr_port_FIHEL'
        ]
        
    def fit(self, X, y=None):
        return self
        
    def transform(self, X):
        # Ensure input is a DataFrame
        if isinstance(X, pd.Series):
            df = X.to_frame().T
        else:
            df = X.copy()
        
        # A. Time Features (From Notebook 04)
        if 'updated_ts' in df.columns:
            df['updated_ts'] = pd.to_datetime(df['updated_ts'], errors='coerce')
            if 'imo' in df.columns:
                df = df.sort_values(by=['imo', 'updated_ts'])
            
            df['day_of_week'] = df['updated_ts'].dt.dayofweek
            # Time deltas per vessel
            if 'imo' in df.columns:
                df['time_since_last_position_min'] = df.groupby('imo')['updated_ts'].diff().dt.total_seconds() / 60.0
            else:
                df['time_since_last_position_min'] = 0.0
            df['reporting_interval_min'] = df['time_since_last_position_min']
        
        # B. Telemetry: Rolling & Deltas
        if 'sog' in df.columns:
            if 'imo' in df.columns:
                df['rolling_mean_sog'] = df.groupby('imo')['sog'].transform(lambda x: x.rolling(3, min_periods=1).mean())
                df['speed_delta'] = df.groupby('imo')['sog'].diff()
            else:
                df['rolling_mean_sog'] = df['sog']
                df['speed_delta'] = 0.0
        
        if 'hdg' in df.columns:
            df['heading_change'] = df.groupby('imo')['hdg'].diff() if 'imo' in df.columns else 0.0
            
        if 'cog' in df.columns:
            if 'imo' in df.columns:
                df['movement_stability'] = df.groupby('imo')['cog'].transform(lambda x: x.rolling(3, min_periods=1).std())
            else:
                df['movement_stability'] = 0.0

        # C. Encodings
        if 'arr_port' in df.columns:
            df['arr_port_FIHEL'] = (df['arr_port'] == 'FIHEL').astype(float)
        
        # D. Schema Alignment & NaN Handling (Crucial for XGBoost)
        for col in self.selected_features:
            if col not in df.columns:
                df[col] = 0.0
            # Ensure all selected features are numeric
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Final cleanup: fill all NaNs with 0 and ensure 2D shape
        transformed_data = df[self.selected_features].fillna(0).astype(float)
        return transformed_data.values.reshape(-1, len(self.selected_features))

# --- 2. CONFIGURATION AND LOADING ---
PROJECT_PATH = "/Users/rober/smartport-ai-risk-early-warning/"
# Using the parent cleaned data (Pre-engineered)
X_train_raw = pd.read_csv(PROJECT_PATH + "02_Data/03_Working/work_clean.csv")
y_train = pd.read_csv(PROJECT_PATH + "02_Data/03_Working/y_balanced.csv").squeeze()

# Aligning X with the balanced Y indices
X_train_final = X_train_raw.loc[X_train_raw.index.isin(y_train.index)]

# --- 3. PIPELINE ASSEMBLY ---
pipeline = Pipeline(steps=[
    ('engineer', SmartPortFeatureEngineer()), 
    ('imputer', SimpleImputer(strategy='median')), 
    ('model', XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42))
])

# --- 4. TRAINING AND EXPORT ---
print("🚀 Training production pipeline with internal transformations...")
pipeline.fit(X_train_final, y_train)

MODEL_PATH = PROJECT_PATH + "04_Models/smartport_model.pkl"
joblib.dump(pipeline, MODEL_PATH)
print(f"✅ Success: Encapsulated model saved to {MODEL_PATH}")

/var/folders/6c/byy38myn1t50449jbzfjs1_c0000gn/T/ipykernel_2029/2482065580.py:80: DtypeWarning: Columns (0: updated_ts, 1: etd_schedule, 2: etd, 3: eta_schedule, 4: eta, 5: ata) have mixed types. Specify dtype option on import or set low_memory=False.
  X_train_raw = pd.read_csv(PROJECT_PATH + "02_Data/03_Working/work_clean.csv")


🚀 Training production pipeline with internal transformations...


/Users/rober/ai-corporate-suite/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [11:37:47] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ Success: Encapsulated model saved to /Users/rober/smartport-ai-risk-early-warning/04_Models/smartport_model.pkl


## Automated Model Retraining & Quality Audit

In [10]:
import cloudpickle
from sklearn.metrics import recall_score

def run_retraining():
    print("Starting automated retraining process...")
    
    # 1. Load existing execution pipeline
    model_path = os.path.join(MODELS_PATH, 'pipe_execution.pkl')
    with open(model_path, 'rb') as f:
        pipe = cloudpickle.load(f)
    
    # 2. Re-fit pipeline on current balanced data
    print("Refitting model on new data...")
    pipe.fit(X, y)
    
    # 3. Quality Audit: Ensure Recall (Sensitivity) is above 90%
    # In maritime risk, catching delays is more critical than false alarms
    y_pred = pipe.predict(X)
    recall = recall_score(y, y_pred)
    
    if recall >= 0.90:
        with open(model_path, 'wb') as f:
            cloudpickle.dump(pipe, f)
        print(f"🚀 Retraining Successful! Model updated with Recall: {recall:.4f}")
    else:
        print(f"⚠️ Update Rejected: Recall {recall:.4f} is below the 0.90 threshold.")

if __name__ == "__main__":
    run_retraining()

Starting automated retraining process...
Refitting model on new data...


/Users/rober/ai-corporate-suite/venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [11:37:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


🚀 Retraining Successful! Model updated with Recall: 0.9646


## PRODUCTION EXECUTION SCRIPT (SCALED FOR BUSINESS)

In [11]:
import os
import pandas as pd
import cloudpickle
from datetime import datetime

def run_execution():
    print("🚀 Starting Production Execution...")
    
    # --- Configuration ---
    PROJECT_PATH = '/Users/rober/smartport-ai-risk-early-warning'
    MODEL_FILE = os.path.join(PROJECT_PATH, '04_Models/pipe_execution.pkl')
    FEATURES_FILE = os.path.join(PROJECT_PATH, '04_Models/model_features.pkl')
    SOURCE_DATA = os.path.join(PROJECT_PATH, '02_Data/03_Working/work_fs.csv')
    OUTPUT_CSV = os.path.join(PROJECT_PATH, '05_Outputs/risk_alerts.csv')

    # --- Load Model and Features ---
    with open(MODEL_FILE, 'rb') as f:
        pipe = cloudpickle.load(f)
    with open(FEATURES_FILE, 'rb') as f:
        expected_features = cloudpickle.load(f)
    
    # --- Data Loading and Alignment ---
    df_live = pd.read_csv(SOURCE_DATA)
    X_live = df_live.reindex(columns=expected_features, fill_value=0).astype('float32')
    
    # 1. Get raw probabilities from the XGBoost model
    raw_scores = pipe.predict_proba(X_live)[:, 1]
    
    # 2. Build Results DataFrame and Sort by Risk (Ranking)
    results = pd.DataFrame({'vessel_id': df_live.index, 'raw_score': raw_scores})
    results = results.sort_values(by='raw_score', ascending=False)
    
    # 3. FORCED CATEGORIZATION STRATEGY (Operational Calibration)
    # Initialize all records as NORMAL
    results['risk_score'] = 0.15
    results['risk_level'] = 'NORMAL'
    
    # Assign WARNING status (Rank positions 101 to 1000)
    results.iloc[100:1000, results.columns.get_loc('risk_score')] = 0.65
    results.iloc[100:1000, results.columns.get_loc('risk_level')] = 'WARNING'
    
    # Assign CRITICAL status (Top 100 offenders)
    results.iloc[0:100, results.columns.get_loc('risk_score')] = 0.92
    results.iloc[0:100, results.columns.get_loc('risk_level')] = 'CRITICAL'
    
    # --- Action Mapping and Metadata ---
    ACTION_MAP = {
        'CRITICAL': 'Immediate intervention (reassign berth)',
        'WARNING': 'Monitor ETA and AIS stability closely',
        'NORMAL': 'Routine operations'
    }
    results['recommended_action'] = results['risk_level'].map(ACTION_MAP)
    results['timestamp'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    # --- Export Results ---
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    results.to_csv(OUTPUT_CSV, index=False)
    
    print(f"✅ Execution Complete.")
    print(f"📊 Distribution check: {results['risk_level'].value_counts().to_dict()}")

if __name__ == "__main__":
    run_execution()

🚀 Starting Production Execution...
✅ Execution Complete.
📊 Distribution check: {'NORMAL': 115481, 'WARNING': 900, 'CRITICAL': 100}
